# Notebook 03: Machine Learning Regressors & Mass-Balance Reconciled Ensemble
**Models Covered**:
1. **Random Forest Regressor** (Regime-aware bagged nonlinear threshold trees on stationary target deltas)
2. **Gradient Boosting Regressor** (Stagewise additive boosted trees with Huber-like shrinkage)
3. **Mass-Balance Reconciled Hybrid Ensemble** (Industry-grade upgrade combining Gradient Boosting, Random Forest, Exponential Smoothing, and physical queue conservation `Care(t) = Care(t-1) + Transfers(t) - Discharges(t)`)

### Industry-Grade Capabilities Demonstrated
- **Regime-Normalized Walk-Forward Training**: Prevents high-volume 2023–2024 variance from distorting low-volume 2025 predictions.
- **Conformal Prediction Intervals (80% & 95%)**: Calibrated uncertainty fans across multi-step horizons.
- **Explainable AI (Feature Attribution)**: Quantifies the contribution of lag, rolling mean, and net-flow pressure signals.

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
    HAS_SKLEARN = True
except ImportError:
    HAS_SKLEARN = False

from app import load_and_prepare_data, engineer_features, run_walk_forward_backtest, generate_multi_horizon_forecasts

df = load_and_prepare_data("../data/HHS_Unaccompanied_Alien_Children_Program.csv", "data/HHS_Unaccompanied_Alien_Children_Program.csv", "HHS_Unaccompanied_Alien_Children_Program.csv")
feat = engineer_features(df)

lb_care, pred_care, imp_care = run_walk_forward_backtest(feat, target="hhs_care", test_days=180)
lb_disc, pred_disc, imp_disc = run_walk_forward_backtest(feat, target="hhs_discharged", test_days=180)

ml_models = ["Random Forest Regressor", "Mass-Balance Reconciled Ensemble", "Gradient Boosting Regressor"]
print("=== MACHINE LEARNING & ENSEMBLE MODELS: CHILDREN IN HHS CARE ===")
print(lb_care.loc[ml_models, ["Family", "MAE", "RMSE", "MAPE (%)", "Forecast Accuracy (%)", "Horizon 7d MAE", "Horizon 14d MAE", "Horizon 30d MAE", "Forecast Stability Index", "Model Robustness"]].to_string())
print("\n=== MACHINE LEARNING & ENSEMBLE MODELS: DISCHARGE DEMAND ===")
print(lb_disc.loc[ml_models, ["Family", "MAE", "RMSE", "MAPE (%)", "Forecast Accuracy (%)", "Horizon 7d MAE", "Horizon 14d MAE", "Horizon 30d MAE", "Forecast Stability Index", "Model Robustness"]].to_string())

=== MACHINE LEARNING & ENSEMBLE MODELS: CHILDREN IN HHS CARE ===
                                             Family   MAE  RMSE  MAPE (%)  Forecast Accuracy (%)  Horizon 7d MAE  Horizon 14d MAE  Horizon 30d MAE  Forecast Stability Index  Model Robustness
Model                                                                                                                                                                                             
Random Forest Regressor            Machine Learning  4.61  5.94      0.21                  99.79            5.85             6.44             7.34                      98.7              99.8
Mass-Balance Reconciled Ensemble  Hybrid Reconciled  5.09  6.53      0.23                  99.77            6.46             7.11             8.11                      99.4              99.8
Gradient Boosting Regressor        Machine Learning  5.35  6.86      0.24                  99.76            6.79             7.47             8.52                     

In [2]:
# Top 10 Predictive Feature Drivers (Explainable AI Attribution)
top_features = pd.Series(imp_care).sort_values(ascending=False).head(10)
print("Top 10 Predictive Features for Children in HHS Care (% Attribution):")
print(top_features.to_string())

plt.figure(figsize=(10, 4.5))
top_features.sort_values().plot(kind="barh", color="#2563eb")
plt.title("Top 10 Predictive Feature Drivers (Children in HHS Care)")
plt.xlabel("Relative Attribution Importance (%)")
plt.tight_layout()
plt.show()

Top 10 Predictive Features for Children in HHS Care (% Attribution):
care_momentum_7d                  11.42
net_flow_roll_mean_7               9.85
transferred_to_hhs_roll_mean_7     8.74
hhs_care_roll_std_7                7.91
cbp_custody_lag_1                  7.18
net_flow_lag_1                     6.82
hhs_care_lag_1                     6.45
hhs_care_lag_2                     6.12
hhs_discharged_roll_mean_7         5.89
transferred_to_hhs_lag_1           5.41